<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/04_agentic_ai.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Building an AI Course Agent with the OpenAI Agents SDK</h2>
            <h4 style="color:#00bf00;">Copyright 2026 Dr Elena Kramer, Braude College, Israel.</h4>
            <span style="color:#00bfff;">In this lab, we turn the course assistant from Lab 3 into an agent. The OpenAI Agents SDK runs the agent loop and the tool calls for us, while GPT-OSS:120B runs on our Ollama server. We add Phoenix tracing, an email-recording tool, and a Gradio chat interface.</span>
        </td>
    </tr>
</table>

Set `OLLAMA_BASE_URL` and `PHOENIX_API_KEY` in the project-root `.env` file, and run the notebook from top to bottom.

## What you will learn

In Lab 3 we built the course assistant by hand: we wrote the tool's JSON schema, checked `finish_reason`, ran the tool ourselves and sent the result back to the model in a loop. Here we build the **same assistant** with the OpenAI Agents SDK and see what the framework does for us:

- **Tools** - `@function_tool` builds the tool description from a regular Python function.
- **The agent loop** - `Runner.run` calls the model, runs the tool, and returns the result to the model until there is a final answer.
- **Decisions** - the model decides when to use each tool, based on the instructions. This is what makes it an agent and not just an LLM call.
- **Several steps** - with two tools, the agent first checks whether an email is already saved, and only then saves it: several tool calls in one run, chained by the agent itself.
- **Memory** - `SQLiteSession` keeps the conversation, so the chat interface needs only a few lines.
- **Observability** - Phoenix shows what the agent actually did: the LLM calls, the tool calls and their inputs.

In [ ]:
import os
from urllib.parse import urlparse

import gradio as gr
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pypdf import PdfReader
from agents import Agent, OpenAIChatCompletionsModel, Runner, function_tool, SQLiteSession
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor
from phoenix.otel import register

## 1. Connect to the model and to Phoenix

Ollama speaks the OpenAI API, so we create an `AsyncOpenAI` client that points at it and wrap it in `OpenAIChatCompletionsModel`. The API key `"ollama"` is just a placeholder.

Phoenix runs on the same server as Ollama, on port `6006`. Open `http://SERVER_IP:6006`, log in (user: `student@braude.ac.il`, password: `1234567890`), create a project named with **your student id**, and put that id below.

In [ ]:
load_dotenv(override=True)

MODEL = "gpt-oss:120b"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
PHOENIX_API_KEY = os.getenv("PHOENIX_API_KEY")

# Phoenix: same server as Ollama, port 6006
server = urlparse(OLLAMA_BASE_URL).hostname
tracer_provider = register(
    project_name="023605248",          # <---------- your student id
    endpoint=f"http://{server}:6006/v1/traces",
    api_key=PHOENIX_API_KEY,
)
OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)

# The model
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
agent_model = OpenAIChatCompletionsModel(model=MODEL, openai_client=ollama_client)

## 2. Load the course material

The same two documents as in Lab 3: a text summary and the syllabus PDF.

In [ ]:
summary = open("docs/summary.txt", encoding="utf-8").read()

reader = PdfReader("docs/syllabus.pdf")
syllabus = ""
for page in reader.pages:
    syllabus += page.extract_text() or ""

print(f"Summary: {len(summary)} characters, syllabus: {len(syllabus)} characters")

## 3. The agent's instructions

Instructions describe the agent's role and rules (this is the system prompt).  
Notice the rule about emails: we simply tell the agent **when** to use the tool, and the model decides.

In [ ]:
instructions = f"""
You are the digital assistant for the AI Agents Course. Be professional, friendly
and helpful. Answer questions about the course using these materials:

COURSE SUMMARY
{summary}

SYLLABUS
{syllabus}

Rules:
- If the answer is not in the materials, say that you don't know.
- You are an AI assistant; don't claim to be a human.
- If the user shares an email address, first use the check_email tool to see
  whether it is already saved. If it is not saved, save it with the record_email tool.
  If the user shares several addresses, check and save each one separately.
  If a tool says an address is not valid, tell the user.
  If the user asks you not to save it, don't.
"""

## 4. Give the agent tools

In Lab 3 we wrote the JSON schema of the tool by hand. Here the `@function_tool` decorator builds it from the function name, the type hints and the docstring.

We give the agent **two** tools: one that reads the `emails.txt` file and one that writes to it.

The LLM chooses the arguments of a tool call, and it can make mistakes - for example pass `"t2"` instead of an email address. So `record_email` does one simple check, and if the address is not valid it **returns a message to the agent** instead of saving it. The agent reads that message and can tell the user.

In [ ]:
@function_tool
def check_email(email: str) -> str:
    """Check whether the given email address is already saved."""
    try:
        saved = open("emails.txt", encoding="utf-8").read().split()
    except FileNotFoundError:
        saved = []
    if email in saved:
        return "This email is already saved"
    return "This email is not saved yet"


@function_tool
def record_email(email: str) -> str:
    """Save one email address so the course team can contact the user."""
    email = email.strip()
    if "@" not in email or "." not in email.split("@")[-1]:
        return f"'{email}' is not a valid email address, so it was not saved"
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    print(f"Saved email: {email}")
    return "The email was saved"

In [ ]:
agent = Agent(
    name="AI Course Assistant",
    instructions=instructions,
    model=agent_model,
    tools=[check_email, record_email],
)

## 5. Run the agent

`Runner.run` is the agent loop: it calls the model, runs the tool if the model asks for it, sends the result back to the model, and returns the final answer.  
No `finish_reason` checks and no hand-written loop like in Lab 3.

In [ ]:
result = await Runner.run(agent, "Hello! What is this course about?")
print(result.final_output)

In [ ]:
result = await Runner.run(agent, "My email is student@example.com")
print(result.final_output)

In [ ]:
# The same email again - the agent should find it with check_email and not save it twice
result = await Runner.run(agent, "Hi, my email is student@example.com")
print(result.final_output)

In [ ]:
# Several addresses, one of them broken
result = await Runner.run(agent, "Please save t1@gmail.com t1@gmail com t2@gmail.com")
print(result.final_output)

In [ ]:
result = await Runner.run(agent, "My email is other@example.com, but please don't save it")
print(result.final_output)

In [ ]:
# What was saved?
print(open("emails.txt", encoding="utf-8").read())

Now open Phoenix and look at the traces. In the first email run you should see **two** tool calls: `check_email` and then `record_email`. In the second run - only `check_email`.  
Which tools did the agent call when we asked it not to save the email?

## 6. Add a chat interface

A `SQLiteSession` keeps the conversation history for us: every call to `Runner.run` with the same session sees the previous messages.  
So our `chat` function only needs to pass the new message.

In [ ]:
session = SQLiteSession("course_chat")

async def chat(message, history):
    result = await Runner.run(agent, message, session=session)
    return result.final_output

In [ ]:
gr.ChatInterface(fn=chat, title="AI Course Assistant").launch(inbrowser=True)

# Congratulations!

You built a course assistant with the OpenAI Agents SDK, connected it to GPT-OSS:120B on Ollama, gave it a tool, traced it in Phoenix, and launched a chat interface.

Compared with Lab 3, the SDK runs the agent loop and the tool calls; we only write the Python function and the instructions.